In [54]:
### This script is used to analyze the 2p data and corresponding treadmill behavior data
### 1. Load the 2p data and treadmill behavior data
### 2. Align the 2p data and treadmill behavior data
### 3a. Smooth the deconvolved traces using a 250 ms Gaussian window
### 3b. Remove inactive data points
### 4. Spatial discretization (divide the VR corridor into ~110 bins, each representing 1cm and assign each data point to its corresponding spatial bin)
### 5. Test for reliability for individual cells (calculate Pearson CC or cohen’s D)
### 6. Plotting activity of all responsive cells (cross validation – split trials in half)
### 7. Population vector cross-correlation 

In [55]:
%cd "C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation"

C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation


**1. Load the 2p data and treadmill behavior data**

In [56]:
import os
import re
import datetime
import numpy as np
from helper import TwoP, read_xml, time2float
import matplotlib.pyplot as plt

# File path to VRlog.txt and 2p data
filepath = r'D:\V1_SpatialModulation\2p\250320_JSY_JSY041_spatialmodulation\TSeries-03202025-1804-002'
VRfilepath = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog"
twoP_filename = "TSeries-03202025-1804-002"
VRlog_filename = "VRlog_JSY038_03202025_06-56-38.txt"
VRlog_path = os.path.join(VRfilepath, VRlog_filename)

# Extract animal ID and date from the VR_log_filename
match = re.match(r"VRlog_(JSY\d+)_(\d{8})_\d{2}-\d{2}-\d{2}\.txt", VRlog_filename)
if match:
    animal_id = match.group(1)
    date = match.group(2)
else:
    print("Filename format does not match the expected pattern.")

# Initialize dictionaries to store raw data
twoP_data = {}
VR_data = {}

# Load twoP data
raw_twop_data = TwoP(filepath, twoP_filename)

raw_twop_data.find_files()
twop_dict = raw_twop_data.calc_dFF()

twoP_data['sps'] = twop_dict['spikes_per_sec'].copy()
twoP_data['s2p_spks'] = twop_dict['s2p_spks'].copy()
twoP_data['dFF'] = twop_dict['norm_dFF'].copy()

numFrames = np.size(twoP_data['sps'], 1)

xml_path = os.path.join(filepath, f"{twoP_filename}.xml")
xml_dict = read_xml(xml_path)
t0 = xml_dict["t0"]
abs_time = xml_dict["abs_time"]
rel_time = xml_dict["rel_time"]
framerate = 1/rel_time[1]
print(framerate)

twopT = np.zeros(np.size(abs_time, 0) - 1, dtype=datetime.datetime)
for rep, t in enumerate(abs_time[:-1]):
    twopT[rep] = t0 + datetime.timedelta(seconds=t)

twopT_float = time2float(twopT)
twoP_data['AbsoluteT'] = twopT

# # print the data type of twoP_data['twopT'[0]]
# print(type(twoP_data['AbsoluteT'][0]))


# Load VRlog
rawVR_data = []
with open(VRlog_path, "r") as file:
    lines = file.readlines()
    for line in lines[3:]:
        rawVR_data.append(line.strip().split("\t"))
        
# Extract VR data
VR_data['absoluteT'] = np.array([line[0] for line in rawVR_data])
VR_data['elapsedT'] = np.array([float(line[1]) for line in rawVR_data])
VR_data['event'] = np.array([line[2] for line in rawVR_data])
VR_data['location'] = np.array([float(line[3]) for line in rawVR_data])

# Find the index of the first 's' in VR_data['event']
start_index = np.where(VR_data['event'] == 's')[0][0]

# Erase all elements before the start_index in all VR_data
for key in VR_data.keys():
    VR_data[key] = VR_data[key][start_index:]

# # for every element of VR_data, print first value
# for key in VR_data:
#     print(VR_data[key][0])

print("first time value of VR is", VR_data['absoluteT'][0])
print("first time value of 2p is", twoP_data['AbsoluteT'][0] )

c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\oasis\functions.py:817: FutureWarning: Beginning in SciPy 1.17, multidimensional input will be treated as a batch, not `ravel`ed. To preserve the existing behavior and silence this warning, `ravel` arguments before passing them to `toeplitz`.
  A = scipy.linalg.toeplitz(xc[np.arange(lags)],


KeyboardInterrupt: 

**2. Align the 2p data and treadmill behavior data**

In [ ]:
# Define absolute_t0 as the first element of VR_data['absoluteT'] -- with "s" for event type, which is the timestamp for 2p input trigger
VR_absolute_t = np.array([datetime.datetime.strptime(t, '%H.%M.%S.%f') for t in VR_data['absoluteT'][0:]])

# Calculate relative_t (time elapsed from absolute_t0)
VR_relative_t = np.array([(t - VR_absolute_t[0]).total_seconds() for t in VR_absolute_t])

# Add twoP_data['AbsoluteT'][0] to each timedelta object to get vrT
VR_relative_t_timedelta = np.array([datetime.timedelta(seconds=t) for t in VR_relative_t])
Aligned_Abs_vrT = twoP_data['AbsoluteT'][0] + VR_relative_t_timedelta

# Find the closest value in Aligned_Abs_vrT that is greater than twoP_data['AbsoluteT'][-1]
closest_value = Aligned_Abs_vrT[Aligned_Abs_vrT > twoP_data['AbsoluteT'][-1]][0]
closest_index = np.where(Aligned_Abs_vrT == closest_value)[0][0]

# print(closest_value)
# print(closest_index)

new_VR_data = {}
new_VR_data['AbsoluteT'] = np.array(Aligned_Abs_vrT)[:closest_index]
new_VR_data['RelativeT'] = VR_relative_t[:closest_index]
new_VR_data['event'] = VR_data['event'][:closest_index]
new_VR_data['location'] = VR_data['location'][:closest_index]

print(new_VR_data['RelativeT'][-1])
new_VR_data['location'][new_VR_data['location'] < -460] = -460

# Calculate relative time points for VR_data and twoP_data
twop_relativeT = twoP_data['AbsoluteT'] - twoP_data['AbsoluteT'][0]

# Convert to seconds
twop_relativeT = np.array([t.total_seconds() for t in twop_relativeT])
twoP_data['RelativeT'] = twop_relativeT

# print("start of 2p is", twoP_data['RelativeT'][0])
# print("start of VR is", new_VR_data['RelativeT'][0])

# print("end of 2p is", twoP_data['RelativeT'][-1])
# print("end of VR is", new_VR_data['RelativeT'][-1])

# Interpolate the location at twoP_data['RelativeT'] from new_VR_data['location'] at new_VR_data['RelativeT']
interpolated_location = np.interp(twoP_data['RelativeT'], new_VR_data['RelativeT'], new_VR_data['location'])
new_VR_data['interp_location'] = interpolated_location

# # for all other keys in new_VR_data, cut off the last few elements to match the length of interpolated_location
# for key in new_VR_data.keys():
#     new_VR_data[key] = new_VR_data[key][:len(interpolated_location)]

# # Now interpolated_location contains the interpolated location values at twop_relativeT
# print(new_VR_data['location'].shape)
# print(twoP_data['RelativeT'].shape)
# print(interpolated_location.shape)

# Plot the interpolated location
plt.figure(figsize=(20, 5))
plt.plot(twoP_data['RelativeT'], new_VR_data['interp_location']+100, label="Interpolated Location", alpha=0.5)
plt.plot(new_VR_data['RelativeT'], new_VR_data['location'], label="Original Location", alpha=0.5)
plt.legend()
plt.show()

# print(interpolated_location[0:10])
# print(aligned_VR_data['aligned_location'][0:10])

# plt.figure(figsize=(20, 5))
# plt.plot(twoP_data['RelativeT'], twoP_data['dFF'][10, :])
# plt.show()

# plt.figure(figsize=(20, 5))
# plt.plot(twoP_data['RelativeT'], twoP_data['sps'][10, :])
# plt.show()

**3a. Smooth the deconvolved traces using a 250 ms Gaussian window**

In [ ]:
from helper import SpikeSmoothing, BehavioralDataFiltering as DF, spatial_discretization as SD, ReliabilityTesting as RT

# Define offsets to test (in frames)
# have a list of offsets to test from -10 to 10 with 1 increment
# offset_frames_list = list(range(-10, 11, 2))
offset_frames_list = [-10, -5, -3, -2, -1, 0, 1, 2, 3, 5, 10]
framerate = 1/rel_time[1]  # Use the same framerate as before

# Dictionary to store results
offset_results = {}

# Loop through offsets
for offset_frames in offset_frames_list:
    print(f"\n\nTesting offset: {offset_frames} frames ({offset_frames/framerate:.2f} seconds)")
    
    # Apply offset to original data
    offset_spike_data = SpikeSmoothing.apply_temporal_offset(twoP_data['sps'], offset_frames)
    
    # Apply smoothing
    smoothed = SpikeSmoothing.smooth_spikes(offset_spike_data, framerate, window_ms=500)
    
    # Process data with trial filtering
    filtered_spks_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
        smoothed, 
        new_VR_data['interp_location'],
        min_trial_duration_seconds=5, 
        max_trial_duration_seconds=30,
        framerate=framerate
    )
    
    if n_valid_laps == 0:
        print(f"No valid laps for offset {offset_frames}")
        continue
    
    
    single_revolution_VR = 282.415
    single_revolution_treadmill = 27.8
    single_lap_VR = 1126.0667 ### = 1146 when VR length was 125 at gain = 1.15 
    single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR


    # Perform spatial assignment
    spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
        n_valid_laps,
        filtered_spks_laps, 
        filtered_location_laps, 
        single_lap_treadmill
    )
    
    # Apply spatial smoothing
    smoothed_spatial_activity = SpikeSmoothing.spatial_smooth(spatial_activity, window_cm=10)


    # Test for reliability
    reliable_cells, avg_cc, cohens_d, iter_cc, _ = RT.test_cell_reliability(
        smoothed_spatial_activity,
        n_shuffles=100,           
        cc_percentile=95,          
        cohen_threshold=1.2,       
        min_cc_threshold=0.3,      
        min_activity_threshold=0.05, 
    )

    # Store results
    offset_results[offset_frames] = {
        'reliable_cells': reliable_cells,
        'reliable_count': np.sum(reliable_cells),
        'avg_cc': avg_cc,
        'cohens_d': cohens_d,
        'spatial_activity': smoothed_spatial_activity,
        'n_valid_laps': n_valid_laps
    }

    # Print summary
    print(f"Offset {offset_frames}: Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
    print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
    print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")
    
 # Extract metrics for visualization
valid_offsets = list(offset_results.keys())
reliable_counts = [offset_results[offset]['reliable_count'] for offset in valid_offsets]
avg_cc_means = [np.mean(offset_results[offset]['avg_cc'][offset_results[offset]['reliable_cells']]) 
               if np.sum(offset_results[offset]['reliable_cells']) > 0 else 0
               for offset in valid_offsets]
cohens_d_means = [np.mean(offset_results[offset]['cohens_d'][offset_results[offset]['reliable_cells']])
                 if np.sum(offset_results[offset]['reliable_cells']) > 0 else 0
                 for offset in valid_offsets]

# Create figure
fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

# Plot reliable cell count
axes[0].plot(valid_offsets, reliable_counts, 'o-', color='blue', linewidth=2)
axes[0].set_ylabel('Number of Reliable Cells')
axes[0].set_title('Effect of Temporal Offset on Cell Reliability Metrics')
axes[0].grid(True, alpha=0.3)

# Plot mean correlation coefficient
axes[1].plot(valid_offsets, avg_cc_means, 'o-', color='green', linewidth=2)
axes[1].set_ylabel('Mean Correlation Coefficient')
axes[1].grid(True, alpha=0.3)

# Plot mean Cohen's D
axes[2].plot(valid_offsets, cohens_d_means, 'o-', color='red', linewidth=2)
axes[2].set_xlabel('Offset (frames)')
axes[2].set_ylabel("Mean Cohen's D")
axes[2].grid(True, alpha=0.3)

# Find optimal offset (if results exist)
if len(valid_offsets) > 0:
    # Normalize metrics for weighted average
    norm_reliable = np.array(reliable_counts) / np.max(reliable_counts) if np.max(reliable_counts) > 0 else np.zeros_like(reliable_counts)
    norm_cc = np.array(avg_cc_means) / np.max(avg_cc_means) if np.max(avg_cc_means) > 0 else np.zeros_like(avg_cc_means)
    norm_d = np.array(cohens_d_means) / np.max(cohens_d_means) if np.max(cohens_d_means) > 0 else np.zeros_like(cohens_d_means)
    
    # Weighted sum of normalized metrics
    combined_metric = (norm_reliable + norm_cc + norm_d) / 3
    best_idx = np.argmax(combined_metric)
    best_offset = valid_offsets[best_idx]
    
    # Add vertical line at optimal offset
    for ax in axes:
        ax.axvline(x=best_offset, color='black', linestyle='--', alpha=0.7)
        ax.text(best_offset, ax.get_ylim()[1]*0.95, f'Optimal: {best_offset}', 
               horizontalalignment='center', verticalalignment='top')
    
    print(f"\nBest offset: {best_offset} frames ({best_offset/framerate:.2f} seconds)")
    print(f"- Reliable cells: {offset_results[best_offset]['reliable_count']}")
    print(f"- Mean correlation: {np.mean(offset_results[best_offset]['avg_cc'][offset_results[best_offset]['reliable_cells']]):.3f}")
    print(f"- Mean Cohen's D: {np.mean(offset_results[best_offset]['cohens_d'][offset_results[best_offset]['reliable_cells']]):.3f}")

plt.tight_layout()
plt.show()   

In [ ]:
import numpy as np
from helper import SpikeSmoothing

offset_spike_data = SpikeSmoothing.apply_temporal_offset(twoP_data['sps'], best_offset)
smoothed = SpikeSmoothing.smooth_spikes(offset_spike_data, framerate, window_ms=500)
# smoothed = SpikeSmoothing.smooth_spikes(twoP_data['sps'], framerate, window_ms=500)
twoP_data['smoothed_spks'] = smoothed
SpikeSmoothing.plot_comparison(twoP_data['s2p_spks'], smoothed, framerate)

**3b. Remove inactive data points**

In [ ]:
### 1) any time points where the treadmill behavior data is not available (i.e. when the animal is not moving)
### 2) any trial that is taking longer than 30 seconds to complete

from helper import BehavioralDataFiltering as DF
filtered_spks_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
    twoP_data['smoothed_spks'], 
    new_VR_data['interp_location'],
    min_trial_duration_seconds=5, 
    max_trial_duration_seconds=30,
    framerate=framerate
)

**4. Discretize spatial data (1cm/bin) and assign 2p data to a corresponding bin**

In [ ]:
from helper import spatial_discretization as SD
from helper import SpikeSmoothing

# # when VR length was 300 at gain = 1.15 - 25/03/20 
single_revolution_VR = 282.415
single_revolution_treadmill = 27.8
single_lap_VR = 1126.0667 ### = 1146 when VR length was 125 at gain = 1.15 
single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# # when VR length was 125 at gain = 1.15 
# single_revolution_VR = 285.9317
# single_revolution_treadmill = 27.8
# single_lap_VR = 1146 ### = 1146 when VR length was 125 at gain = 1.15 
# single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# Then perform spatial assignment on the filtered data
spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
    n_valid_laps,
    filtered_spks_laps, 
    filtered_location_laps, 
    single_lap_treadmill
)

smoothed_spatial_activity = SpikeSmoothing.spatial_smooth(spatial_activity, window_cm=10)


**5. Test for reliability for individual cells (calculate Pearson CC or cohen’s D)**

In [ ]:
from helper import ReliabilityTesting as RT
# Run the analysis
reliable_cells, avg_cc, cohens_d, iter_cc, _ = RT.test_cell_reliability(
    smoothed_spatial_activity,
    n_shuffles=100,           # Use 1000+ for final analysis
    cc_percentile=95,          # 90th percentile threshold for CC
    cohen_threshold=1.2,       # Medium-large effect size
    min_cc_threshold=0.3,      # Minimum correlation required
    min_activity_threshold=0.05 # Minimum activity level (relative to max)
)

# Print summary
print(f"Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")

normalized_spatial_activity = RT.normalize_spatial_activity(smoothed_spatial_activity)

In [ ]:
# # Option 1.1 Plot the average activity oin a grid layout
# reliable_avg = {}

# for i in np.where(reliable_cells)[0]:
#     reliable_avg[i] = np.mean(spatial_activity[i, :, :], axis=0)

# # plot average activity of first 25 reliable cells in a 5 x 5 subplots
# fig, axs = plt.subplots(5, 5, figsize=(20, 20))
# reliable_indices = list(reliable_avg.keys())[:25]
# for idx, i in enumerate(reliable_indices):
#     ax = axs[idx // 5, idx % 5]
#     ax.plot(bin_centers, reliable_avg[i])
#     ax.set_title(f'Cell {i}')
#     ax.set_xlabel('Position')
#     ax.set_ylabel('Activity')

# # set a title over all plots
# fig.suptitle('Average spikes of example reliable cells', fontsize=16)
# plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust the rect parameter to add space at the top
# plt.show()

# # Option 1.2: Plot heatmaps in a grid layout (eg. 20 cells in a 5×4 grid)
# fig1 = RT.plot_reliable_cells_grid(
#     normalized_spatial_activity,
#     reliable_cells,
#     max_cells=20,                # Show up to 20 reliable cells
#     avg_cc=avg_cc,               # Optional correlation coefficients
#     cohen_d=cohens_d,            # Optional Cohen's D values
#     normalize=False,              # Apply normalization
#     n_rows=5,                    # 5 rows
#     n_cols=4                     # 4 columns
# )
# plt.suptitle('Spikes of reliable cells', fontsize=15)
# plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top

# plt.show()

# Option 2: Plot with both heatmap and trial-averaged activity
fig2 = RT.plot_reliable_cells_side_by_side(
    normalized_spatial_activity,
    reliable_cells,
    max_cells=50,                # Show up to 10 reliable cells
    avg_cc=avg_cc,               # Optional correlation coefficients
    cohen_d=cohens_d,            # Optional Cohen's D values
    normalize=False               # Apply normalization
)
plt.suptitle('Spikes of reliable cells', fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.985])  # Adjust the rect parameter to add space at the top
plt.show()

# # Option 3: Plot waterfall plots
# RT.plot_reliable_cells_waterfall(normalized_spatial_activity, reliable_cells, max_cells=np.sum(reliable_cells))
# plt.show()

**6. Plotting activity of all responsive cells (cross validation – split trials in half)**

In [ ]:
from helper import ResponseVisualization as RV
# fig1, sorted_indices = RV.create_response_plot(spatial_activity, reliable_cells)  # Optional: manually set contrast limits for stronger effect

fig1, sorted_indices = RV.create_response_plot(normalized_spatial_activity, reliable_cells, clim=(0, 1))  # Optional: manually set contrast limits for stronger effect
# draw a vertical line on a plot

# # For waterfall visualization (alternative approach):
# fig2 = RV.create_waterfall_plot(
#     normalized_spatial_activity,
#     reliable_cells
# )

plt.show()

**7. Spatial Modulation Index (SMI)**

In [ ]:
from helper import SpatialModulationIndex as spatialmodulationI
import numpy as np

# First, let's normalize and scale the bin centers to match the actual physical distance
original_bin_centers = bin_centers.copy()

# Step 1: Shift to start at 0
shifted_centers = bin_centers - np.min(bin_centers)

# Step 2: Scale to match the actual physical distance (0 to 110cm)
actual_corridor_length = np.size(spatial_bins)  # cm
unity_corridor_length = np.max(shifted_centers)
scaled_bin_centers = shifted_centers * (actual_corridor_length / unity_corridor_length)

print(f"Original bin centers range: {np.min(original_bin_centers):.1f} to {np.max(original_bin_centers):.1f}")
print(f"Scaled bin centers range: {np.min(scaled_bin_centers):.1f} to {np.max(scaled_bin_centers):.1f}")

# Set parameters
segment_distance = 51.8  # Distance between visually identical positions
exclude_boundary_cm = 3.67  # Border to exclude

# Calculate SMI with properly scaled bin centers
results = spatialmodulationI.calculate_SMI(normalized_spatial_activity, scaled_bin_centers, 
                           segment_distance=segment_distance, 
                           exclude_boundary_cm=exclude_boundary_cm)

# Plot the results
spatialmodulationI.plot_SMI_results(results, scaled_bin_centers, reliable_cells=reliable_cells, max_examples=50)
# plt.show()

results = spatialmodulationI.analyze_spatial_modulation(normalized_spatial_activity, bin_centers, reliable_cells)
plt.show()

In [ ]:
from helper import SpatialModulationIndex as spatialmodulationI
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# First, normalize and scale the bin centers to match the actual physical distance
original_bin_centers = bin_centers.copy()
shifted_centers = bin_centers - np.min(bin_centers)
actual_corridor_length = 110  # cm
unity_corridor_length = np.max(shifted_centers)
scaled_bin_centers = shifted_centers * (actual_corridor_length / unity_corridor_length)

print(f"Scaled bin centers range: {np.min(scaled_bin_centers):.1f} to {np.max(scaled_bin_centers):.1f}")

# Set parameters
segment_distance = 55  # Distance between visually identical positions
exclude_boundary_cm = 3  # Border to exclude

# Calculate SMI with properly scaled bin centers
results = spatialmodulationI.calculate_SMI(normalized_spatial_activity, scaled_bin_centers, 
                           segment_distance=segment_distance, 
                           exclude_boundary_cm=exclude_boundary_cm)

# Extract the SMI values for valid cells
SMI_values = results['SMI']
valid_cells = results['valid_cells']
valid_SMI = SMI_values[valid_cells]

# Remove any NaN or Inf values if present
valid_SMI = valid_SMI[~np.isnan(valid_SMI) & ~np.isinf(valid_SMI)]

print(f"Number of cells with valid SMI: {len(valid_SMI)}")

# Calculate summary statistics
median_SMI = np.median(valid_SMI)
mad_SMI = stats.median_abs_deviation(valid_SMI)
print(f"Median SMI ± MAD: {median_SMI:.2f} ± {mad_SMI:.2f}")

# Statistical test (Wilcoxon signed-rank test against 0)
stat, p_value = stats.wilcoxon(valid_SMI)
print(f"Wilcoxon test against SMI=0: p-value = {p_value:.2e}")
print(f"Significantly {'different from' if p_value < 0.05 else 'not different from'} 0")

# Create cumulative distribution plot with full SMI range (including negative values)
plt.figure(figsize=(8, 6))
x_sorted = np.sort(valid_SMI)  # Keep original SMI values (including negatives)
y_cumulative = np.arange(1, len(x_sorted) + 1) / len(x_sorted)

plt.plot(x_sorted, y_cumulative, 'k-', linewidth=2)

# Add reference lines
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7)

plt.xlabel('Spatial modulation index')
plt.ylabel('Cumul. probability')
plt.title('Cumulative Distribution of Spatial Modulation Index')
plt.xlim(-1, 1)
plt.ylim(0, 1)
plt.grid(False)
plt.tight_layout()
plt.show()

# Print proportion of cells with different modulation patterns
prop_positive = np.mean(valid_SMI > 0)
prop_negative = np.mean(valid_SMI < 0)
prop_strong_pos = np.mean(valid_SMI > 0.5)
print(f"Proportion of cells with positive modulation (SMI > 0): {prop_positive:.2f} ({prop_positive*100:.1f}%)")
print(f"Proportion of cells with negative modulation (SMI < 0): {prop_negative:.2f} ({prop_negative*100:.1f}%)")
print(f"Proportion of cells with strong positive modulation (SMI > 0.5): {prop_strong_pos:.2f} ({prop_strong_pos*100:.1f}%)")

**7. Population vector cross-correlation**